# Exercises
(Date - 24/July/2026)

In [1]:
import torch
import torchvision
import sys
from pathlib import Path
import importlib
sys.path.append(str(Path.cwd().parent))
from helper_functions import set_seeds, download_data, plot_loss_curves
from going_modular import data_setup, engine, utils, predict
importlib.reload(predict)
import matplotlib.pyplot as plt
from torchvision import transforms
from torch import nn
from torchinfo import summary
import random
from summary_writer import create_writer
from helper_functions import plot_loss_curves, download_data
import requests
from vit import ViT as imported_ViT
from torchvision.models import vit_b_16, ViT_B_16_Weights


device = "cuda" if torch.cuda.is_available() else "cpu"
device


c:\Users\user\AppData\Local\Programs\Python\Python311\Lib\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


'cpu'

1. Replicate the ViT architecture we created with in-built PyTorch transformer layers.
* You'll want to look into replacing our TransformerEncoderBlock() class with torch.nn.TransformerEncoderLayer() (these contain the same layers as our custom blocks).
* You can stack torch.nn.TransformerEncoderLayer()'s on top of each other with torch.nn.TransformerEncoder().

In [2]:
class TransformerEncoderBlock(nn.Module):
    def __init__(self,
                 hidden_size=768,
                 num_heads=12,
                 mlp_size=3072,
                 dropout=0.1):
        super().__init__()

        self.block = nn.TransformerEncoderLayer(
            d_model=hidden_size,
            nhead=num_heads,
            dim_feedforward=mlp_size,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

    def forward(self, x):
        return self.block(x)

TransformerEncoderBlock_ = TransformerEncoderBlock()

In [3]:
summary(model=TransformerEncoderBlock_,
        input_size=(1,197,768),
        col_names=["input_size", "output_size", "num_params", "trainable"],
        col_width=20,
        row_settings=["var_names"])

Layer (type (var_name))                            Input Shape          Output Shape         Param #              Trainable
TransformerEncoderBlock (TransformerEncoderBlock)  [1, 197, 768]        [1, 197, 768]        --                   True
├─TransformerEncoderLayer (block)                  [1, 197, 768]        [1, 197, 768]        --                   True
│    └─LayerNorm (norm1)                           [1, 197, 768]        [1, 197, 768]        1,536                True
│    └─MultiheadAttention (self_attn)              [1, 197, 768]        [1, 197, 768]        2,362,368            True
│    └─Dropout (dropout1)                          [1, 197, 768]        [1, 197, 768]        --                   --
│    └─LayerNorm (norm2)                           [1, 197, 768]        [1, 197, 768]        1,536                True
│    └─Linear (linear1)                            [1, 197, 768]        [1, 197, 3072]       2,362,368            True
│    └─Dropout (dropout)                     

In [4]:
class PatchEmbedding(nn.Module):
    """Turns a 2D input image into a 1D sequence learnable embedding vector.

    Args:
        in_channels (int): Number of color channels for the input images. Defaults to 3.
        patch_size (int): Size of patches to convert input image into. Defaults to 16.
        embedding_dim (int): Size of embedding to turn image into. Defaults to 768.
    """

    def __init__(self,
                 in_channels=3,
                 patch_size=16,
                 embedding_dim=768):
        super().__init__()
        
        self.patcher = nn.Conv2d(in_channels=in_channels,
                                 out_channels=embedding_dim,
                                 kernel_size=patch_size,
                                 stride=patch_size,
                                 padding=0)
        
        self.flatten = nn.Flatten(start_dim=2,
                                  end_dim=3)
        
    def forward(self, x):
        image_resolution = x.shape[-1]
        assert image_resolution % self.patch_size == 0, f"Image size must be divisible by patch size | Image shape: {image_resolution} | Patch size: {self.patch_size}"

        x_patched = self.patcher(x)
        x_flatten = self.flatten(x_patched)
        return x_flatten.permute(0,2,1)

In [5]:
class ViT(nn.Module):
    def __init__(self,
                 img_size:int=224,
                 patch_size:int=16,
                 in_channels:int=3,
                 num_layers:int=12,
                 hidden_size_D:int=768,
                 MLP_size:int=3072,
                 num_heads:int=12,
                 attn_dropout:float=0,
                 mlp_dropout:float=0.1,
                 embedding_dropout:float=0.1,
                 weight_decay:float=0.03,
                num_classes:int=1000
                 ):
        super().__init__()
        assert img_size % patch_size == 0, f"Image size is not divisible by patch size. Image size: {img_size} | Patch size: {patch_size}"
        self.num_patches = int((img_size**2)/(patch_size**2))
        self.class_embedding = nn.Parameter(data=torch.randn(1, 1, hidden_size_D), requires_grad=True)
        self.pos_embedding = nn.Parameter(data=torch.randn(1, self.num_patches+1, hidden_size_D), requires_grad=True)
        self.embedding_dropout = nn.Dropout(p=embedding_dropout)
        self.patch_embedding = PatchEmbedding(in_channels=in_channels,
                                              patch_size=patch_size,
                                              embedding_dim=hidden_size_D)
        self.Transformer_encoder = nn.Sequential(
                    *[
                        TransformerEncoderBlock(
                            hidden_size=hidden_size_D,
                            num_heads=num_heads,
                            mlp_size=MLP_size,
                            dropout=mlp_dropout
                        )
                        for _ in range(num_layers)
                    ]
                )
        self.classifier = nn.Sequential(
            nn.LayerNorm(normalized_shape=hidden_size_D),
            nn.Linear(in_features=hidden_size_D,
                      out_features=num_classes)
        )

    def forward(self, x):
        batch_size = x.shape[0]
        # 13. Create class token embedding and expand it to match the batch size (equation 1)
        class_token = self.class_embedding.expand(batch_size, -1, -1) # "-1" means to infer the dimension (try this line on its own)

        # 14. Create patch embedding (equation 1)
        x = self.patch_embedding(x)

        # 15. Concat class embedding and patch embedding (equation 1)
        x = torch.cat((class_token, x), dim=1)

        # 16. Add position embedding to patch embedding (equation 1)
        x = self.pos_embedding + x

        # 17. Run embedding dropout (Appendix B.1)
        x = self.embedding_dropout(x)

        # 18. Pass patch, position and class embedding through transformer encoder layers (equations 2 & 3)
        x = self.Transformer_encoder(x)

        # 19. Put 0 index logit through classifier (equation 4)
        x = self.classifier(x[:, 0]) # run on each sample in a batch at 0 index

        return x

(Date - 25/July/2026)

2. Turn the custom ViT architecture we created into a Python script, for example, vit.py.
* You should be able to import an entire ViT model using something likefrom vit import ViT.

In [6]:
%%writefile vit.py

import torch
from torch import nn
class TransformerEncoderBlock(nn.Module):
    def __init__(self,
                 hidden_size=768,
                 num_heads=12,
                 mlp_size=3072,
                 dropout=0.1):
        super().__init__()

        self.block = nn.TransformerEncoderLayer(
            d_model=hidden_size,
            nhead=num_heads,
            dim_feedforward=mlp_size,
            dropout=dropout,
            activation="gelu",
            batch_first=True,
            norm_first=True
        )

    def forward(self, x):
        return self.block(x)

class PatchEmbedding(nn.Module):
    """Turns a 2D input image into a 1D sequence learnable embedding vector.

    Args:
        in_channels (int): Number of color channels for the input images. Defaults to 3.
        patch_size (int): Size of patches to convert input image into. Defaults to 16.
        embedding_dim (int): Size of embedding to turn image into. Defaults to 768.
    """

    def __init__(self,
                 in_channels=3,
                 patch_size=16,
                 embedding_dim=768):
        super().__init__()
        self.patch_size=patch_size
        
        self.patcher = nn.Conv2d(in_channels=in_channels,
                                 out_channels=embedding_dim,
                                 kernel_size=patch_size,
                                 stride=patch_size,
                                 padding=0)
        
        self.flatten = nn.Flatten(start_dim=2,
                                  end_dim=3)
        
    def forward(self, x):
        image_resolution = x.shape[-1]
        assert image_resolution % self.patch_size == 0, f"Image size must be divisible by patch size | Image shape: {image_resolution} | Patch size: {self.patch_size}"

        x_patched = self.patcher(x)
        x_flatten = self.flatten(x_patched)
        return x_flatten.permute(0,2,1)

class ViT(nn.Module):
    def __init__(self,
                 img_size:int=224,
                 patch_size:int=16,
                 in_channels:int=3,
                 num_layers:int=12,
                 hidden_size_D:int=768,
                 MLP_size:int=3072,
                 num_heads:int=12,
                 attn_dropout:float=0,
                 mlp_dropout:float=0.1,
                 embedding_dropout:float=0.1,
                 weight_decay:float=0.03,
                num_classes:int=1000
                 ):
        super().__init__()
        assert img_size % patch_size == 0, f"Image size is not divisible by patch size. Image size: {img_size} | Patch size: {patch_size}"
        self.num_patches = int((img_size**2)/(patch_size**2))
        self.class_embedding = nn.Parameter(data=torch.randn(1, 1, hidden_size_D), requires_grad=True)
        self.pos_embedding = nn.Parameter(data=torch.randn(1, self.num_patches+1, hidden_size_D), requires_grad=True)
        self.embedding_dropout = nn.Dropout(p=embedding_dropout)
        self.patch_embedding = PatchEmbedding(in_channels=in_channels,
                                              patch_size=patch_size,
                                              embedding_dim=hidden_size_D)
        self.Transformer_encoder = nn.Sequential(*[TransformerEncoderBlock() for _ in range(num_layers)])
        self.classifier = nn.Sequential(
            nn.LayerNorm(normalized_shape=hidden_size_D),
            nn.Linear(in_features=hidden_size_D,
                      out_features=num_classes)
        )

    def forward(self, x):
        batch_size = x.shape[0]
        # 13. Create class token embedding and expand it to match the batch size (equation 1)
        class_token = self.class_embedding.expand(batch_size, -1, -1) # "-1" means to infer the dimension (try this line on its own)

        # 14. Create patch embedding (equation 1)
        x = self.patch_embedding(x)

        # 15. Concat class embedding and patch embedding (equation 1)
        x = torch.cat((class_token, x), dim=1)

        # 16. Add position embedding to patch embedding (equation 1)
        x = self.pos_embedding + x

        # 17. Run embedding dropout (Appendix B.1)
        x = self.embedding_dropout(x)

        # 18. Pass patch, position and class embedding through transformer encoder layers (equations 2 & 3)
        x = self.Transformer_encoder(x)

        # 19. Put 0 index logit through classifier (equation 4)
        x = self.classifier(x[:, 0]) # run on each sample in a batch at 0 index

        return x


Overwriting vit.py


In [7]:
imported_ViT

vit.ViT

In [8]:
Vision_transformer = imported_ViT()
Vision_transformer

ViT(
  (embedding_dropout): Dropout(p=0.1, inplace=False)
  (patch_embedding): PatchEmbedding(
    (patcher): Conv2d(3, 768, kernel_size=(16, 16), stride=(16, 16))
    (flatten): Flatten(start_dim=2, end_dim=3)
  )
  (Transformer_encoder): Sequential(
    (0): TransformerEncoderBlock(
      (block): TransformerEncoderLayer(
        (self_attn): MultiheadAttention(
          (out_proj): NonDynamicallyQuantizableLinear(in_features=768, out_features=768, bias=True)
        )
        (linear1): Linear(in_features=768, out_features=3072, bias=True)
        (dropout): Dropout(p=0.1, inplace=False)
        (linear2): Linear(in_features=3072, out_features=768, bias=True)
        (norm1): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (norm2): LayerNorm((768,), eps=1e-05, elementwise_affine=True, bias=True)
        (dropout1): Dropout(p=0.1, inplace=False)
        (dropout2): Dropout(p=0.1, inplace=False)
      )
    )
    (1): TransformerEncoderBlock(
      (block): T

(Date - 26/July/2026)

3. Train a pretrained ViT feature extractor model (like the one we made in 08. PyTorch Paper Replicating section 10) on 20% of the pizza, steak and sushi data like the dataset we used in 07. PyTorch Experiment Tracking section 7.3.
* See how it performs compared to the EffNetB2 model we compared it to in 08. PyTorch Paper Replicating section 10.6.

In [9]:
loss_fn = torch.nn.CrossEntropyLoss()

optimizer = torch.optim.Adam(params=Vision_transformer.parameters(),
                             lr = 0.003,
                             betas=(0.9, 0.999),
                             weight_decay=0.3)

In [10]:
train_dir="data/pizza_steak_sushi/train"
test_dir="data/pizza_steak_sushi/test"

transform = transforms.Compose([
    transforms.Resize(size=(224,224)),
    transforms.ToTensor()
])

train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(train_dir=train_dir,
                                                                  test_dir=test_dir,
                                                                  batch_size=1,
                                                                  train_transform=transform,
                                                                  test_transform=transform
                                                                )
train_dataloader, test_dataloader, class_names

(<torch.utils.data.dataloader.DataLoader at 0x1f84e352110>,
 ['pizza', 'steak', 'sushi'])

In [11]:
results = engine.train(model=Vision_transformer,
                       loss_fn=loss_fn,
                       optimizer=optimizer,
                       epochs=1,
                       device=device,
                       writer=create_writer(experiment_name="ViT_exercise",
                                            model_name="ViT_base"),
                       train_dataloader=train_dataloader,
                       test_dataloader=test_dataloader
                                            
    )

results

[INFO] Created SummaryWriter, saving to: runs\2026-07-28\ViT_exercise\ViT_base...


100%|██████████| 1/1 [10:49<00:00, 649.10s/it]

Epoch: 1 | train_loss: 1.7959 | train_acc: 0.3467 | test_loss: 1.2506 | test_acc: 0.4133


{'train_loss': [1.7958944844620095],
 'train_acc': [0.3466666666666667],
 'test_loss': [1.2505869301160177],
 'test_acc': [0.41333333333333333]}

We'll do only one epoch... sorry!

(Date - 27/July 2026)

4. Try repeating the steps from excercise 3 but this time use the "ViT_B_16_Weights.IMAGENET1K_SWAG_E2E_V1" pretrained weights from torchvision.models.vit_b_16().
* Note: ViT pretrained with SWAG weights has a minimum input image size of (384, 384) (the pretrained ViT in exercise 3 has a minimum input size of (224, 224)), though this is accessible in the weights .transforms() method.

In [12]:
weights = ViT_B_16_Weights.IMAGENET1K_SWAG_E2E_V1
model = vit_b_16(weights=weights)

In [13]:
summary(model=model)

Layer (type:depth-idx)                                            Param #
VisionTransformer                                                 768
├─Conv2d: 1-1                                                     590,592
├─Encoder: 1-2                                                    443,136
│    └─Dropout: 2-1                                               --
│    └─Sequential: 2-2                                            --
│    │    └─EncoderBlock: 3-1                                     7,087,872
│    │    └─EncoderBlock: 3-2                                     7,087,872
│    │    └─EncoderBlock: 3-3                                     7,087,872
│    │    └─EncoderBlock: 3-4                                     7,087,872
│    │    └─EncoderBlock: 3-5                                     7,087,872
│    │    └─EncoderBlock: 3-6                                     7,087,872
│    │    └─EncoderBlock: 3-7                                     7,087,872
│    │    └─EncoderBlock: 3-8         

In [14]:
for params in model.parameters():
    params.requires_grad = True

In [15]:
summary(model=model)

Layer (type:depth-idx)                                            Param #
VisionTransformer                                                 768
├─Conv2d: 1-1                                                     590,592
├─Encoder: 1-2                                                    443,136
│    └─Dropout: 2-1                                               --
│    └─Sequential: 2-2                                            --
│    │    └─EncoderBlock: 3-1                                     7,087,872
│    │    └─EncoderBlock: 3-2                                     7,087,872
│    │    └─EncoderBlock: 3-3                                     7,087,872
│    │    └─EncoderBlock: 3-4                                     7,087,872
│    │    └─EncoderBlock: 3-5                                     7,087,872
│    │    └─EncoderBlock: 3-6                                     7,087,872
│    │    └─EncoderBlock: 3-7                                     7,087,872
│    │    └─EncoderBlock: 3-8         

In [16]:
manual_transform = transforms.Compose([
    transforms.Resize(size=(384,384)),
    transforms.ToTensor()
])

train_dataloader, test_dataloader, class_names = data_setup.create_dataloaders(train_dir=train_dir,
                                                                  test_dir=test_dir,
                                                                  batch_size=1,
                                                                  train_transform=manual_transform,
                                                                  test_transform=manual_transform
                                                                )
train_dataloader, test_dataloader, class_names

(<torch.utils.data.dataloader.DataLoader at 0x1f851d53f50>,
 ['pizza', 'steak', 'sushi'])

In [17]:
results_0 = engine.train(model=model,
                       loss_fn=loss_fn,
                       optimizer=optimizer,
                       epochs=1,
                       device=device,
                       writer=create_writer(experiment_name="ViT_exercise",
                                            model_name="ViT_base"),
                       train_dataloader=train_dataloader,
                       test_dataloader=test_dataloader
                                            
    )

results_0

[INFO] Created SummaryWriter, saving to: runs\2026-07-28\ViT_exercise\ViT_base...


100%|██████████| 1/1 [19:13<00:00, 1153.43s/it]

Epoch: 1 | train_loss: 8.1836 | train_acc: 0.0000 | test_loss: 8.1710 | test_acc: 0.0000


{'train_loss': [8.183595330980088],
 'train_acc': [0.0],
 'test_loss': [8.171045500437419],
 'test_acc': [0.0]}

(Date - 28/July/2026)

5. Our custom ViT model architecture closely mimics that of the ViT paper, however, our training recipe misses a few things. Research some of the following topics from Table 3 in the ViT paper that we miss and write a sentence about each and how it might help with training:
* ImageNet-21k pretraining (more data).
* Learning rate warmup.
* Learning rate decay.
* Gradient clipping.

Answer:

1. ImageNet-21k pretraining:<br>
ImageNet-1k has 1.3 million images and 1000 classes while ImageNet-21k has 14 million images and 21,000 classes. Unlike CNNs, which are biased towards image recognition (i.e. they know about the relation between pixels, edges, corners,curves, etc.), ViTs are completely blank. Therefore, they require pretraining to be able to work properly.

2. Learning rate warmup:<br>
As the training starts, the learning rate slowly increases, i.e. the model takes big steps. <br>
For example:<br>
epoch 1: LR = 0.001<br>
epoch 2: LR = 0.002<br>
epoch 3: LR = 0.003<br>
...

3. Learning rate decay:<br>
It is just opposite to learning rate warmup, i.e., as the training is going on, the learning rate slowly decreases, i.e. the model takes short steps.

4. Gradient clipping:<br>
While training, sometimes the gradients get very big which can explode the computation. Therefore, gradient clipping regularly fix these huge gradients by scaling them down.